# Predicting arrears next month
Follows `protocol.yaml`: temporal holdout from `holdout.csv`, walk-forward CV on training rows, selection by mean CV ROC AUC, one scoring on holdout.

**Feature decision:** `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. it is not known at prediction time (and has an ~87% arrears rate, a classic leak). It is excluded. `meter_ref` and `billing_month` are excluded per protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==20939 and len(hold_idx)==7536
is_hold = np.zeros(len(df), bool); is_hold[hold_idx]=True
train = df[~is_hold]; hold = df[is_hold]
TARGET='arrears_next_month'
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM
print(len(train), len(hold), train.billing_month.max(), hold.billing_month.min())
print(train[TARGET].mean(), hold[TARGET].mean())

## Candidates (named before any scoring)

In [ ]:
def prep(scale):
    return ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('num',StandardScaler() if scale else 'passthrough',NUM)])
candidates = {
 'baseline_prior': Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'hgb_depth3': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
 'hgb_depth6': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

## Walk-forward CV on training rows (expanding window over billing months, 5 splits)

In [ ]:
months = np.sort(train.billing_month.unique())
chunks = np.array_split(months, 6)   # first chunk = initial training window; each later chunk is a validation fold
folds=[]
for k in range(1,6):
    tr_m = np.concatenate(chunks[:k]); va_m = chunks[k]
    tr = np.where(train.billing_month.isin(tr_m))[0]; va = np.where(train.billing_month.isin(va_m))[0]
    folds.append((tr,va))
train_pos = np.where(~is_hold)[0]   # positions in data.csv
cv_scores={}
for name,est in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(train.iloc[tr][FEATURES], train.iloc[tr][TARGET])
        s.append(float(roc_auc_score(train.iloc[va][TARGET], m.predict_proba(train.iloc[va][FEATURES])[:,1])))
    cv_scores[name]=s
cv_table=pd.DataFrame(cv_scores).T; cv_table['mean']=cv_table.mean(axis=1); print(cv_table.round(4))

## Choose by highest mean CV ROC AUC, refit on all training rows, score holdout once

In [ ]:
chosen = max(cv_scores, key=lambda n: np.mean(cv_scores[n]))
final = clone(candidates[chosen]).fit(train[FEATURES], train[TARGET])
holdout_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1]))
print(chosen, holdout_auc)

In [ ]:
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[train_pos[tr].tolist(), train_pos[va].tolist()] for tr,va in folds],
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': hold_idx.tolist(),
 'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))

## Result
The headline is the holdout ROC AUC printed above for the CV-chosen candidate, without the leaky `final_notice_sent`. The holdout is later in time than all training data, so it estimates performance on future months.